# MvDEC: canonical REUTERS rebuild fix and View2 latent-20 probe

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Two checks:

1. `scripts/make_reuters_resample.py` rebuilds REUTERS-10K from the RCV1-v2 token files, now permuting the whole single-label corpus with seed 1234 before taking the first 10,000 documents. `resample_summary.json` holds its agreement with the DEKM release arrays; S2 (seed 42) then runs once on the rebuilt sample.
2. `scripts/mvdec_view2_latent_probe.py` pretrains View2 with a 20-d latent (seeds 42-44) beside the frozen DEKM_F seed-42 View1 and fuses them by weighted concatenation. Weight 0.5 is label-free; the best weight is a label-guided ceiling (`latent20_probe.json`).

Kaggle: GPU on, Internet on, attach dataset `datnguyends/mvdec-v1-true-bottleneck-largest-eigen-s42-input`. The download is about 400 MB.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PINNED_COMMIT = "7acc399cad4fc97b74f62516da297b4e6bde93be"
BRANCH = "claude/mvdec-final-round"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/temp/mvdec-canonical-rebuild-latent20")
RESULTS = Path("/kaggle/working/mvdec-canonical-rebuild-latent20")
P = "mvdec_2025_"
S2 = P + "scratch_both_views_true_bottleneck_min500_release_l2_v1"
SAMPLE_SEEDS = ()
RUNS = [
    {"group": sample, "protocol": S2, "dataset": "REUTERS", "seed": 42,
     "dataset_root": f"/kaggle/temp/reuters_resample/{sample}"}
    for sample in ("canonical",)
]


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path in (REPO, OUTPUT, RESULTS):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
(RESULTS / "trajectory").mkdir(parents=True)
(RESULTS / "logs").mkdir()
subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)
if any(run["dataset"] == "20NEWS" for run in RUNS):
    # The official DEKM 20NEWS path reads the scikit-learn cache without downloading.
    from sklearn.datasets import fetch_20newsgroups

    print("20NEWS documents:", len(fetch_20newsgroups(subset="all").data))
EXPECTED_CHECKPOINT_MANIFEST_SHA256 = "71e3c2d423d70bbde1b7423c91c176c1d33a4fb473e7cb6febec187fff8f0607"
EXPECTED_CHECKPOINT_WEIGHT_SHA256 = "ca55a1e995697c346d99540ddd9cc8bd8e6182a7d21d0b09410d7e76b6050702"
candidates = sorted(
    manifest.parent
    for manifest in Path("/kaggle/input").rglob("manifest.json")
    if (manifest.parent / "final.weights.h5").is_file()
    and file_sha256(manifest) == EXPECTED_CHECKPOINT_MANIFEST_SHA256
)
assert len(candidates) == 1, candidates
CHECKPOINT_MANIFEST = candidates[0] / "manifest.json"
assert file_sha256(candidates[0] / "final.weights.h5") == EXPECTED_CHECKPOINT_WEIGHT_SHA256
print("DEKM_F seed-42 View1 checkpoint:", candidates[0])


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
        "tests/test_mvdec_trajectory.py",
        "tests/test_make_reuters_resample.py",
        "tests/test_mvdec_view2_latent_probe.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)

In [ ]:
RESAMPLE_ROOT = Path("/kaggle/temp/reuters_resample")
with (RESULTS / "logs" / "make_reuters_resample.log").open("w", encoding="utf-8") as log:
    resample = subprocess.run(
    [
        sys.executable, "scripts/make_reuters_resample.py",
        "--raw-dir", "/kaggle/temp/rcv1_tokens",
        "--output-root", str(RESAMPLE_ROOT),
        "--release-dir", "external_repos/DEKM/datasets/REUTERS",
        *[arg for seed in SAMPLE_SEEDS for arg in ("--sample-seed", str(seed))],
    ],
    cwd=REPO,
    check=False,
    stdout=log,
    stderr=subprocess.STDOUT,
    )
print("make_reuters_resample exit code", resample.returncode, flush=True)
if (RESAMPLE_ROOT / "summary.json").exists():
    shutil.copy2(RESAMPLE_ROOT / "summary.json", RESULTS / "resample_summary.json")
RUNS = [run for run in RUNS if (Path(run["dataset_root"]) / "REUTERS" / "10k_feature.npy").exists()]
print("runs with rebuilt data:", len(RUNS))

In [ ]:
for run in RUNS:
    tag = f"{run['group']}_{run['dataset']}_seed_{run['seed']}"
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", run["dataset"],
        "--runs", "1", "--seed", str(run["seed"]),
        "--protocol", run["protocol"],
        *run.get("extra", []),
        "--progress-interval", "100",
        "--dataset-root", run.get("dataset_root", "external_repos/DEKM/datasets"),
        "--public-output-dir", str(OUTPUT / run["group"]),
    ]
    print("Running:", " ".join(command), flush=True)
    env = {**os.environ, "MVDEC_LABEL_TRAJECTORY_LOG": str(RESULTS / "trajectory" / f"{tag}.jsonl")}
    with (RESULTS / "logs" / f"{tag}.log").open("w", encoding="utf-8") as log:
        result = subprocess.run(
            command, cwd=REPO, env=env, check=False, timeout=4 * 60 * 60,
            stdout=log, stderr=subprocess.STDOUT,
        )
    print(tag, "exit code", result.returncode, flush=True)

In [ ]:
probe = subprocess.run(
    [
        sys.executable, "-u", "scripts/mvdec_view2_latent_probe.py",
        "--view1-weights", str(candidates[0] / "final.weights.h5"),
        "--latent", "20", "--seed", "42", "--seed", "43", "--seed", "44",
        "--output", str(RESULTS / "latent20_probe.json"),
    ],
    cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")},
    stdout=(RESULTS / "logs" / "latent20_probe.log").open("w", encoding="utf-8"),
    stderr=subprocess.STDOUT, check=False,
)
print("latent-20 probe exit code", probe.returncode)
for line in (RESULTS / "logs" / "latent20_probe.log").read_text(encoding="utf-8").splitlines()[-6:]:
    print(line[:400])


In [ ]:
import statistics

PAPER = {
    "REUTERS": {"view1": (0.7642, 0.5999), "view2": (0.6501, 0.5129), "fused": (0.7813, 0.6141)},
    "20NEWS": {"view1": (0.4120, 0.4376), "view2": (0.4086, 0.4032), "fused": (0.4163, 0.4543)},
}
rows = []
for path in sorted(OUTPUT.rglob("manifest.json")):
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("status") != "complete":
        continue
    group = path.relative_to(OUTPUT).parts[0]
    rows.append((group, manifest["dataset"], manifest["seed"], manifest["metrics"]))
    target = RESULTS / "runs" / path.parent.relative_to(OUTPUT)
    target.mkdir(parents=True, exist_ok=True)
    for small in path.parent.iterdir():
        if small.is_file() and small.suffix in {".json", ".csv", ".txt"}:
            shutil.copy2(small, target / small.name)
rows.sort(key=lambda row: (row[0], row[1], row[2]))
print("complete runs:", len(rows), "of", len(RUNS))
for key in sorted({(group, dataset) for group, dataset, _, _ in rows}):
    picked = [(seed, metrics) for group, dataset, seed, metrics in rows if (group, dataset) == key]
    print()
    print(*key)
    for seed, metrics in picked:
        print(seed, "  ".join("%s %.4f/%.4f" % (k, metrics[k + "_acc"], metrics[k + "_nmi"]) for k in PAPER[key[1]]))
    print("rep    mean_ACC best_ACC | paper_ACC | mean_NMI best_NMI | paper_NMI")
    for rep, (paper_acc, paper_nmi) in PAPER[key[1]].items():
        accs = [m[rep + "_acc"] for _, m in picked]
        nmis = [m[rep + "_nmi"] for _, m in picked]
        print("%-6s %8.4f %8.4f | %.4f | %8.4f %8.4f | %.4f" % (rep, statistics.mean(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

zip_path = Path(shutil.make_archive(str(RESULTS) + "-results", "zip", RESULTS.parent, RESULTS.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)